[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/snuuq/ztm-tutorial/blob/main/week3/assignment.ipynb)

> **Colab 사용 시**: 열자마자 `파일 → Drive에 사본 저장`으로 사본을 만들어 작업하세요.
> 이 과제는 GPU 사용을 권장합니다. `런타임 → 런타임 유형 변경 → T4 GPU`에서 바꿀 수 있습니다.
> 노트북이 만드는 파일(`model_comparison.csv`, `confusion_matrix.png`, `misclassified.png`)은 세션이 초기화되기 전에 다운로드하세요.

# 3주차 과제 — KMNIST 모델 세 가지 비교

[KMNIST(Kuzushiji-MNIST)](https://arxiv.org/abs/1812.01718)는 일본 고문서에 등장하는 문자를 모은 이미지 데이터입니다.
MNIST, FashionMNIST와 마찬가지로 28×28 grayscale 이미지와 10개 클래스로 구성되어 있지만, 글자 모양과 분류 대상은 서로 다릅니다.
같은 KMNIST 데이터를 선형 모델, MLP, CNN으로 학습하고 confusion matrix와 오분류 이미지까지 확인합니다.
필요한 구현은 교재 [03장](https://www.learnpytorch.io/03_pytorch_computer_vision/)의 DataLoader, 학습 함수, TinyVGG, 예측 평가 절에서 확인할 수 있습니다.

## 해야 할 일
1. KMNIST를 내려받고 이미지·배치 shape 확인
2. Conv2d와 MaxPool2d를 통과할 때 shape 변화 확인
3. 선형 모델, MLP, TinyVGG 작성
4. 배치 단위 `train_step`, `test_step` 작성
5. 세 모델을 같은 조건에서 학습하고 결과 표 만들기
6. CNN의 confusion matrix와 오분류 샘플 분석

## 규칙
- `# TODO` 부분을 채우세요. 그 외 제공 코드는 수정하지 않아도 됩니다.
- 재현성을 위해 시드는 제공된 값(2525)을 그대로 쓰세요.
- 세 모델은 같은 epoch 수와 optimizer 설정으로 비교하세요.

## 0. 준비

Library import와 device 설정, 시드 고정.

In [ ]:
import time
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(torch.__version__, "| device:", device)

RANDOM_SEED = 2525
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

## 1. KMNIST 다운로드

KMNIST는 28×28 크기의 grayscale 문자 이미지 70,000장으로 구성된 10-class 분류 데이터입니다.
60,000장은 train, 10,000장은 test split으로 제공됩니다.
`transforms.ToTensor()`를 적용하면 이미지 shape는 `[1, 28, 28]`이 됩니다.

In [ ]:
# TODO: transforms.ToTensor()를 만드세요
data_transform = None

# TODO: datasets.KMNIST로 train/test dataset을 만드세요
#       root="data", train=True/False, download=True를 사용하세요.
train_data = None
test_data = None

class_names = train_data.classes
print("train:", len(train_data), "| test:", len(test_data))
print(class_names)

## 2. 이미지 텐서 확인 & 시각화

한 샘플의 shape와 dtype을 확인하세요.
채널이 하나뿐이므로 시각화할 때 `[1, 28, 28]`의 첫 차원을 없애고 grayscale colormap을 사용합니다.

In [ ]:
sample_index = 2525

# TODO: train_data에서 image와 label을 꺼내세요
image, label = None, None

print("image shape:", image.shape)
print("image dtype:", image.dtype)
print("label:", label, class_names[label])

# TODO: channel 차원을 없애고 cmap="gray"로 이미지를 그리세요

## 3. DataLoader

`batch_size=32`로 DataLoader를 만들고 첫 배치가 `[32, 1, 28, 28]`인지 확인하세요.
test DataLoader는 순서를 유지해야 이후 오분류 인덱스를 원본 이미지와 연결할 수 있습니다.

In [ ]:
BATCH_SIZE = 32

# TODO: train/test DataLoader를 만드세요
#       train은 shuffle=True, test는 shuffle=False, num_workers=0을 사용하세요.
train_dataloader = None
test_dataloader = None

# TODO: 첫 train batch를 꺼내 shape를 출력하세요
batch_images, batch_labels = None, None

# print(batch_images.shape, batch_labels.shape)

## 4. Conv2d와 MaxPool2d shape 확인

`dummy batch` 하나를 통과시켜 각 층이 공간 크기를 어떻게 바꾸는지 확인합니다.
`kernel_size=3, stride=1, padding=1`인 convolution은 높이와 너비를 유지합니다.

In [ ]:
torch.manual_seed(RANDOM_SEED)
dummy_batch = torch.randn(1, 1, 28, 28)

# TODO: 1채널을 16채널로 바꾸는 Conv2d를 만드세요
conv = None
# TODO: kernel_size=2인 MaxPool2d를 만드세요
pool = None

# TODO: dummy_batch를 conv, ReLU, pool 순서로 통과시키세요
after_conv = None
after_pool = None

# 확인: [1, 1, 28, 28] → [1, 16, 28, 28] → [1, 16, 14, 14]
# print(dummy_batch.shape, after_conv.shape, after_pool.shape)

## 5. 모델 세 가지 정의

세 모델 모두 입력은 `[batch, 1, 28, 28]`, 출력은 `[batch, 10]`입니다.
TinyVGG는 convolution block 두 개를 사용하고, 각 block 끝에 MaxPool2d를 둡니다.

In [ ]:
class KMNISTLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Flatten → Linear(1*28*28, 10)
        self.layers = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return None


class KMNISTMLPModel(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Flatten → Linear(1*28*28, 256) → ReLU → Linear(256, 10)
        self.layers = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return None


class KMNISTTinyVGG(nn.Module):
    def __init__(self, input_shape=1, hidden_units=16, output_shape=10):
        super().__init__()
        # TODO: Conv2d → ReLU → Conv2d → ReLU → MaxPool2d인 block을 두 개 만드세요
        self.conv_block_1 = None
        self.conv_block_2 = None

        # MaxPool2d를 두 번 거치면 28×28이 7×7이 됩니다.
        # TODO: Flatten과 Linear를 이용해 10개 logits를 출력하세요
        self.classifier = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: 두 convolution block과 classifier를 순서대로 통과시키세요
        return None

## 6. 배치 학습 함수

배치마다 계산한 loss를 단순히 더하지 말고, 배치 크기를 곱해 전체 샘플 기준 평균을 구합니다.
accuracy도 맞힌 샘플 수를 누적한 뒤 dataset 크기로 나누세요.

In [ ]:
def train_step(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        # TODO: forward와 loss 계산
        logits = None
        loss = None

        # TODO: gradient 초기화, backpropagation, parameter update



        # TODO: loss 합계와 맞힌 샘플 수를 누적하세요


    # TODO: dataset의 전체 샘플 수로 나눈 평균 loss와 accuracy(%)를 반환하세요
    return None, None


def test_step(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0

    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            # TODO: logits, loss, 예측 라벨을 계산하고 결과를 누적하세요



    # TODO: dataset의 전체 샘플 수로 나눈 평균 loss와 accuracy(%)를 반환하세요
    return None, None

## 7. 모델 하나를 학습하는 함수

세 모델의 조건을 맞추기 위해 optimizer와 loss를 이 함수 안에서 같은 방식으로 만듭니다.

In [ ]:
def train_model(model, train_dataloader, test_dataloader, epochs, lr, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"epoch": [], "train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    start_time = time.perf_counter()

    for epoch in range(epochs):
        train_loss, train_acc = train_step(
            model, train_dataloader, loss_fn, optimizer, device
        )
        test_loss, test_acc = test_step(
            model, test_dataloader, loss_fn, device
        )

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(
            f"Epoch: {epoch + 1:2d} | "
            f"Train loss: {train_loss:.4f}, acc: {train_acc:5.1f}% | "
            f"Test loss: {test_loss:.4f}, acc: {test_acc:5.1f}%"
        )

    elapsed = time.perf_counter() - start_time
    return history, elapsed

## 8. 세 모델 학습

각 모델을 새로 만들기 직전에 시드를 다시 고정합니다.
모델마다 5 epochs를 학습합니다. KMNIST는 CPU에서도 실행할 수 있지만 GPU나 MPS를 사용하면 더 빠릅니다.

In [ ]:
EPOCHS = 5
LEARNING_RATE = 0.001

model_builders = {
    "Linear": lambda: KMNISTLinearModel(),
    "MLP": lambda: KMNISTMLPModel(),
    "TinyVGG": lambda: KMNISTTinyVGG(),
}

trained_models = {}
model_histories = {}
model_summaries = []

for model_name, build_model in model_builders.items():
    print(f"\n=== {model_name} ===")
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    model = build_model().to(device)
    history, train_seconds = train_model(
        model=model,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        device=device,
    )

    trained_models[model_name] = model
    model_histories[model_name] = history
    model_summaries.append({
        "model": model_name,
        "parameters": sum(p.numel() for p in model.parameters()),
        "train_seconds": train_seconds,
        "test_loss": history["test_loss"][-1],
        "test_accuracy": history["test_acc"][-1],
    })

## 9. 결과 비교 표

모델 이름, 파라미터 수, 학습 시간, test loss, test accuracy를 표로 만들고 CSV로 저장하세요.

In [ ]:
# TODO: model_summaries로 DataFrame을 만들고 test_accuracy 내림차순으로 정렬하세요
comparison = None

# TODO: model_comparison.csv로 index 없이 저장하세요

comparison

## 10. CNN의 전체 test 예측

TinyVGG로 test set 전체를 예측합니다.
test DataLoader를 `shuffle=False`로 만든 이유가 여기서 중요합니다.

In [ ]:
def collect_predictions(model, dataloader, device):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.inference_mode():
        for X, y in dataloader:
            X = X.to(device)

            # TODO: logits에서 예측 라벨을 만드세요
            logits = None
            preds = None

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # TODO: batch별 텐서를 하나로 합쳐 반환하세요
    return None, None


test_preds, test_targets = collect_predictions(
    trained_models["TinyVGG"],
    test_dataloader,
    device,
)

# 확인: torch.Size([10000]) torch.Size([10000])
# print(test_preds.shape, test_targets.shape)

## 11. Confusion matrix 저장

행은 실제 클래스, 열은 예측 클래스를 나타냅니다.
대각선 밖에서 값이 큰 셀은 모델이 자주 헷갈리는 클래스 쌍입니다.

In [ ]:
# TODO: test_targets와 test_preds를 NumPy 배열로 바꿔 confusion matrix를 만드세요
cm = None

fig, ax = plt.subplots(figsize=(10, 10))
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names,
)
display.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("KMNIST confusion matrix — TinyVGG")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
print("그래프 저장: confusion_matrix.png")

## 12. 오분류 샘플 저장

틀린 샘플 중 앞의 9개를 골라 이미지, 예측 라벨, 실제 라벨을 함께 표시하세요.

In [ ]:
# TODO: 예측과 실제 라벨이 다른 위치를 찾으세요
wrong_mask = None
wrong_indices = None

num_to_plot = min(9, len(wrong_indices))
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for plot_index, ax in enumerate(axes.flat):
    if plot_index >= num_to_plot:
        ax.axis(False)
        continue

    dataset_index = int(wrong_indices[plot_index])
    image, true_label = test_data[dataset_index]
    pred_label = int(test_preds[dataset_index])

    ax.imshow(image.squeeze(dim=0), cmap="gray")
    ax.set_title(
        f"Pred: {class_names[pred_label]}\nTrue: {class_names[true_label]}",
        color="red",
    )
    ax.axis(False)

plt.tight_layout()
plt.savefig("misclassified.png", dpi=150, bbox_inches="tight")
print("그래프 저장: misclassified.png")

## 13. 결과 해석

아래 질문에 세 문장 정도로 답하세요.

1. 정확도와 학습 시간을 함께 봤을 때 어떤 모델을 선택하겠나요?
2. confusion matrix에서 가장 자주 헷갈리는 문자 클래스 쌍은 무엇인가요?
3. 오분류 이미지에서 공통적으로 보이는 시각적 특징을 한 가지 이상 찾으세요. 획의 끊김, 굵기, 기울기, 위치 또는 전체 윤곽을 관찰하면 됩니다.

> 답:

## 14. (선택 도전 1) Convolution 설정 바꾸기

4번 섹션에서 만든 `dummy_batch`에 `kernel_size`, `stride`, `padding`을 여러 값으로 바꾼 Conv2d를 적용하세요.
각 설정과 출력 shape를 표로 정리하고, 공간 크기가 달라지는 규칙을 한 문장으로 적어 보세요.

In [ ]:
# (선택) 자유롭게 작성

## 15. (선택 도전 2) CIFAR-10으로 확장하기

CIFAR-10은 32×32 RGB 이미지 60,000장을 10개 클래스로 나눈 데이터입니다.
KMNIST와 달리 입력 채널이 3개이고, pooling을 두 번 거친 공간 크기는 8×8입니다.

CIFAR-10 Python archive는 약 170MB입니다. 다운로드 서버와 네트워크 상태에 따라 상당히 오래 걸릴 수 있으므로 시간 여유가 있을 때만 진행하세요. 한 번 정상적으로 내려받으면 `root` 폴더의 파일을 다시 사용합니다.

1. `datasets.CIFAR10`으로 train/test 데이터를 준비하세요.
2. TinyVGG의 입력 채널을 3으로, classifier 입력을 `hidden_units * 8 * 8`로 바꾸세요.
3. 1~5 epochs만 학습해 KMNIST와 이미지 shape 및 학습 시간을 비교하세요.

아래 셀은 `RUN_CIFAR10_CHALLENGE = True`로 바꾸기 전까지 다운로드되지 않습니다.

In [ ]:
RUN_CIFAR10_CHALLENGE = False

if RUN_CIFAR10_CHALLENGE:
    # TODO: CIFAR-10 train/test dataset을 만드세요
    cifar_train_data = None
    cifar_test_data = None

    # TODO: RGB 32×32 입력에 맞는 TinyVGG를 만들고 짧게 학습하세요

---

## 체크리스트

- [ ] 모든 셀이 위에서 아래로 에러 없이 실행됨
- [ ] 첫 batch shape가 `[32, 1, 28, 28]`
- [ ] Linear, MLP, TinyVGG를 모두 같은 조건에서 학습함
- [ ] `model_comparison.csv`에 파라미터 수와 학습 시간이 포함됨
- [ ] TinyVGG test accuracy가 85% 이상 (권장)
- [ ] `confusion_matrix.png`, `misclassified.png`가 생성됨
- [ ] confusion matrix와 오분류 결과를 세 문장으로 해석함